In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "fermion_ring_1d", "gfmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
using .System1D

default(; dpi=170)
nothing


## Model and GFMC Parameters

This notebook runs fixed-node GFMC for the built-in `SpinlessFermionRing1D` model.

The model represents `N` spinless fermions on a periodic ring with length `L = M * a` and lattice potential
`V(x) = V0 * cos(2*pi*x/a)` applied to each particle coordinate.

Parameters used below:
- Fermion count `N = 2`
- Number of lattice periods `M = 2`
- Lattice spacing `a = 1.0`
- Ring length `L = 2.0`
- Lattice amplitude `V0 = 5.0`
- Twist angle `twist = 0.0`
- Time step `dt = 3.0e-3`
- Total steps `nsteps = 1200`
- Equilibration steps `nequil = 200`
- Target population `targetN = 400`
- Feedback strength `feedback = 0.1`
- Reconfiguration interval `reconfiguration_interval = 2`
- Branch-weight cap `branch_cap = 5.0`
- ET averaging window `energy_window = 40`

Trial / node structure:
- Trial state comes from `trial_wavefunction(model)`
- Guiding path comes from the built-in ring GFMC kernel
- Node policy is `FixedNode()`


## Julia Construction

The next cell constructs the ring model, derives its Hamiltonian and trial data from the source API, and sets the notebook toggles.

This is the cell to edit if you want a different fermion count, twist, debug cadence, or CSV output name.


In [ ]:
N = 100
M = 100
a = 1.0
L = M * a
V0 = -1.0
twist = 0.0

model = SpinlessFermionRing1D(N, a, L, V0; twist=twist, D=0.5, node_tol=1.0e-5, trig_eps=1.0e-10)
H = hamiltonian(model)
trial = trial_wavefunction(model)
guiding = importance_guiding(model)

targetN = 1000
dt = 1e-3
nsteps = 1200
nequil = 200
ET0 = -0.2
feedback = 0.1
reconfiguration_interval = 2
branch_cap = 5.0
energy_window = 40

params = GFMCParams(dt, nsteps, nequil, targetN, ET0, feedback, reconfiguration_interval, branch_cap, energy_window)
RECONFIGURATION = SystematicReconfiguration()

rng_init = MersenneTwister(1234)
initial_positions = sample_uniform_configurations(model, targetN, rng_init)

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
DENSITY_GRID_POINTS = 400
DENSITY_BANDWIDTH = 0.10 * a
PERIOD_MARKERS = collect(0.0:a:L)

RUN_LABEL = "guided fixed node"
RUN_COLOR = :navy
PLOT_TITLE = "Spinless fermion ring GFMC"
DENSITY_TITLE = "Spinless fermion ring GFMC: pooled one-body densities"
UNPOOLED_DENSITY_TITLE = "Spinless fermion ring GFMC: final per-particle one-body densities"
PARTICLE_COLORS = [:navy, :darkorange, :forestgreen, :crimson, :purple, :goldenrod, :deeppink, :teal]

SHOW_PROGRESS = true
PROGRESS_EVERY = 0
DEBUG_MODE = true
DEBUG_EVERY = 1
WRITE_RUN_CSV = false
CSV_FILENAME = "spinless_fermion_ring_gfmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "spinless_fermion_ring_gfmc"


In [ ]:
sim = GFMCSim(
    model,
    params,
    initial_positions,
    MersenneTwister(52);
    use_guiding=true,
    nodepolicy=FixedNode(),
    reconfiguration=RECONFIGURATION,
)
run_gfmc!(
    sim;
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println("final fixed walker count = ", sim.population_history[end])
println(@sprintf("final mean weight = %.6f", sim.mean_weight_history[end]))
println(@sprintf("final effective population = %.2f", sim.effective_population_history[end]))

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_gfmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
function pooled_ring_coordinates(snapshot)
    xs = Float64[]
    for R in snapshot
        append!(xs, Float64.(R))
    end
    return xs
end

function ring_particle_coordinates(snapshot, particle_idx::Integer)
    idx = Int(particle_idx)
    return Float64[R[idx] for R in snapshot]
end

history_fig = nb_plot_gfmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="pooled one-body density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = pooled_ring_coordinates(snapshot)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(density_fig, centers, density; label="step $(step_idx)", color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)

final_snapshot = nb_last_snapshot(sim)
particle_colors = [PARTICLE_COLORS[1 + mod(i - 1, length(PARTICLE_COLORS))] for i in 1:model.N]

unpooled_density_fig = plot(
    xlabel="x",
    ylabel="one-body density",
    title=UNPOOLED_DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for particle_idx in 1:model.N
    xs = ring_particle_coordinates(final_snapshot, particle_idx)
    centers, density = nb_periodic_kde_curve(
        xs;
        xmin=0.0,
        xmax=L,
        grid_points=DENSITY_GRID_POINTS,
        bandwidth=DENSITY_BANDWIDTH,
    )
    plot!(unpooled_density_fig, centers, density; label="particle $(particle_idx)", color=particle_colors[particle_idx], linewidth=2.4)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(unpooled_density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(unpooled_density_fig)
nb_save_figure(unpooled_density_fig, PATHS.figures_dir, FIGURE_STEM, "density_unpooled"; enabled=SAVE_FIGURES)

final_snapshot = nb_last_snapshot(sim)
pair_sep = [distance_1d(model.bc, R[1], R[2]) for R in final_snapshot]

println("minimum separation = ", minimum(pair_sep))
println("count with r < 1e-3 = ", count(r -> r < 1e-3, pair_sep))

sep_fig = histogram(
    pair_sep;
    bins=60,
    normalize=:pdf,
    xlims=(0.0, L / 2),
    xlabel="r = |x1 - x2|",
    ylabel="density",
    title="Pair-separation density",
    label=false,
)
display(sep_fig)

